# 02 — Multiprocessing

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer des processus fils avec `multiprocessing.Process` ;
- utiliser `Pool` pour paralléliser des tâches CPU-bound ;
- échanger des données avec `Queue`, `Pipe` et `Value`/`Array` ;
- utiliser `shared_memory` pour le partage mémoire haute performance ;
- choisir la bonne méthode de démarrage (`fork`, `forkserver`, `spawn`).

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- le module `threading`, `Thread`, `Lock`, `Event` ;
- le GIL et ses limitations pour les tâches CPU-bound ;
- les gestionnaires de contexte (`with`) ;
- les fonctions, closures et higher-order functions ;
- la sérialisation avec `pickle`.

Notions que nous allons **introduire** ici :

- `multiprocessing.Process`, `Pool`, `Queue`, `Pipe` ;
- mémoire partagée (`Value`, `Array`, `shared_memory`) ;
- méthodes de démarrage des processus.

## Plan

1. Pourquoi le multiprocessing ?
2. `Process` — créer un processus fils
3. `Pool` — pool de workers
4. Communication : `Queue` et `Pipe`
5. Mémoire partagée : `Value` et `Array`
6. `shared_memory` — partage haute performance
7. Méthodes de démarrage : `fork`, `forkserver`, `spawn`
8. Pièges courants
9. Synthèse
10. Exercices

---

## 1. Pourquoi le multiprocessing ?

Le **GIL** empêche le vrai parallélisme CPU avec des threads. Le module `multiprocessing` contourne ce problème en créant des **processus séparés**, chacun avec son propre interpréteur Python et son propre GIL.

| Critère | `threading` | `multiprocessing` |
|---|---|---|
| Espace mémoire | Partagé | Séparé |
| GIL | Bloquant pour CPU | Contourné |
| Coût de création | Faible | Élevé |
| Communication | Variables partagées | Sérialisation (pickle) |
| Idéal pour | I/O-bound | CPU-bound |

In [ ]:
import multiprocessing
import os

print(f"Processus principal : PID {os.getpid()}")
print(f"Nombre de CPUs : {multiprocessing.cpu_count()}")

---

## 2. `Process` — créer un processus fils

L'API est très proche de `threading.Thread` : on passe une fonction `target` et des `args`.

In [ ]:
import multiprocessing
import os

def worker(nom: str) -> None:
    print(f"[{nom}] PID={os.getpid()}, Parent PID={os.getppid()}")

if __name__ == "__main__":
    p = multiprocessing.Process(target=worker, args=("fils",))
    p.start()
    p.join()
    print(f"Processus fils terminé, exit code : {p.exitcode}")

**Note importante :** le guard `if __name__ == "__main__"` est **obligatoire** sur Windows et macOS (méthode `spawn`). Sans lui, le processus fils réimporte le module et relance le code, créant une récursion infinie.

### Lancer plusieurs processus

In [ ]:
import multiprocessing
import time

def cpu_bound(n: int) -> int:
    return sum(i * i for i in range(n))

if __name__ == "__main__":
    N = 5_000_000

    # Séquentiel
    start = time.perf_counter()
    for _ in range(4):
        cpu_bound(N)
    t_seq = time.perf_counter() - start

    # Multiprocessing
    start = time.perf_counter()
    procs = [multiprocessing.Process(target=cpu_bound, args=(N,)) for _ in range(4)]
    for p in procs: p.start()
    for p in procs: p.join()
    t_par = time.perf_counter() - start

    print(f"Séquentiel     : {t_seq:.3f}s")
    print(f"Multiprocessing : {t_par:.3f}s")
    print(f"Accélération   : {t_seq / t_par:.1f}x")

### Récupérer un résultat via `Queue`

In [ ]:
import multiprocessing

def calculer(n: int, q: multiprocessing.Queue) -> None:
    q.put(sum(range(n)))

if __name__ == "__main__":
    q = multiprocessing.Queue()
    p = multiprocessing.Process(target=calculer, args=(1_000_000, q))
    p.start()
    resultat = q.get()
    p.join()
    print(f"Résultat : {resultat}")

---

## 3. `Pool` — pool de workers

`Pool` gère automatiquement un groupe de processus workers. C'est la manière la plus simple de paralléliser un `map` sur des données.

In [ ]:
import multiprocessing

def carre(x: int) -> int:
    return x * x

if __name__ == "__main__":
    with multiprocessing.Pool(processes=4) as pool:
        resultats = pool.map(carre, range(10))
    print(resultats)

### `map` vs `imap` vs `imap_unordered`

| Méthode | Retour | Ordre | Mémoire |
|---|---|---|---|
| `map(fn, it)` | `list` | Préservé | Charge tout en mémoire |
| `imap(fn, it)` | Itérateur | Préservé | Lazy |
| `imap_unordered(fn, it)` | Itérateur | Non garanti | Lazy + rapide |

In [ ]:
import multiprocessing
import time

def tache_lente(x: int) -> int:
    time.sleep(0.1)
    return x * x

if __name__ == "__main__":
    with multiprocessing.Pool(4) as pool:
        for result in pool.imap_unordered(tache_lente, range(10)):
            print(f"Reçu : {result}")

### `starmap` — plusieurs arguments

In [ ]:
import multiprocessing

def puissance(base: int, exp: int) -> int:
    return base ** exp

if __name__ == "__main__":
    with multiprocessing.Pool(4) as pool:
        args = [(2, 10), (3, 5), (7, 3), (10, 6)]
        resultats = pool.starmap(puissance, args)
    print(resultats)

### `apply_async` — soumission asynchrone

In [ ]:
import multiprocessing
import time

def calcul_long(x: int) -> int:
    time.sleep(0.5)
    return x * x

if __name__ == "__main__":
    with multiprocessing.Pool(4) as pool:
        futures = [pool.apply_async(calcul_long, (i,)) for i in range(8)]
        for f in futures:
            print(f"Résultat : {f.get(timeout=5)}")

---

## 4. Communication : `Queue` et `Pipe`

Les processus ne partagent pas la mémoire par défaut. Pour communiquer, deux options principales :

### `Queue` — file thread/process-safe

In [ ]:
import multiprocessing

def producteur(q: multiprocessing.Queue) -> None:
    for i in range(5):
        q.put(f"message-{i}")
    q.put(None)  # sentinel

def consommateur(q: multiprocessing.Queue) -> None:
    while True:
        msg = q.get()
        if msg is None:
            break
        print(f"Reçu : {msg}")

if __name__ == "__main__":
    q = multiprocessing.Queue()
    p1 = multiprocessing.Process(target=producteur, args=(q,))
    p2 = multiprocessing.Process(target=consommateur, args=(q,))
    p1.start(); p2.start()
    p1.join(); p2.join()

### `Pipe` — communication bidirectionnelle

In [ ]:
import multiprocessing

def enfant(conn: multiprocessing.Connection) -> None:
    conn.send("Bonjour du processus fils !")
    reponse = conn.recv()
    print(f"[enfant] Reçu : {reponse}")
    conn.close()

if __name__ == "__main__":
    parent_conn, child_conn = multiprocessing.Pipe()
    p = multiprocessing.Process(target=enfant, args=(child_conn,))
    p.start()
    message = parent_conn.recv()
    print(f"[parent] Reçu : {message}")
    parent_conn.send("Bien reçu, merci !")
    p.join()

---

## 5. Mémoire partagée : `Value` et `Array`

`Value` et `Array` permettent de partager des données simples entre processus via la mémoire partagée du système. Les types sont ceux du module `ctypes` (`'i'` = int, `'d'` = double, etc.).

In [ ]:
import multiprocessing

def incrementer(compteur: multiprocessing.Value, lock: multiprocessing.Lock, n: int) -> None:
    for _ in range(n):
        with lock:
            compteur.value += 1

if __name__ == "__main__":
    compteur = multiprocessing.Value('i', 0)  # int partagé
    lock = multiprocessing.Lock()
    procs = [
        multiprocessing.Process(target=incrementer, args=(compteur, lock, 100_000))
        for _ in range(4)
    ]
    for p in procs: p.start()
    for p in procs: p.join()
    print(f"Compteur : {compteur.value} (attendu : 400_000)")

In [ ]:
import multiprocessing

def doubler(arr: multiprocessing.Array) -> None:
    for i in range(len(arr)):
        arr[i] *= 2

if __name__ == "__main__":
    arr = multiprocessing.Array('i', [1, 2, 3, 4, 5])
    p = multiprocessing.Process(target=doubler, args=(arr,))
    p.start()
    p.join()
    print(f"Tableau après : {list(arr)}")

---

## 6. `shared_memory` — partage haute performance

Depuis Python 3.8, `multiprocessing.shared_memory` permet de créer des blocs de mémoire partagée accessibles par nom. C'est plus flexible que `Value`/`Array` et compatible avec NumPy.

In [ ]:
from multiprocessing import shared_memory
import struct

# Créer un bloc de mémoire partagée
shm = shared_memory.SharedMemory(create=True, size=10 * 4)  # 10 ints
print(f"Nom : {shm.name}, taille : {shm.size} octets")

# Écrire des données
for i in range(10):
    struct.pack_into('i', shm.buf, i * 4, i * i)

# Lire les données
valeurs = [struct.unpack_from('i', shm.buf, i * 4)[0] for i in range(10)]
print(f"Valeurs : {valeurs}")

shm.close()
shm.unlink()  # libérer la mémoire

### `ShareableList` — liste partagée simplifiée

In [ ]:
from multiprocessing import shared_memory

sl = shared_memory.ShareableList([0, 1, 2, 3, 4])
print(f"Avant : {list(sl)}")

sl[2] = 99
print(f"Après : {list(sl)}")
print(f"Nom du bloc : {sl.shm.name}")

sl.shm.close()
sl.shm.unlink()

### Utilisation inter-processus avec `shared_memory`

In [ ]:
import multiprocessing
from multiprocessing import shared_memory

def worker_shm(shm_name: str, size: int) -> None:
    """Accède au bloc partagé par nom."""
    shm = shared_memory.SharedMemory(name=shm_name)
    # Modifier le premier octet
    shm.buf[0] = 42
    shm.close()

if __name__ == "__main__":
    shm = shared_memory.SharedMemory(create=True, size=10)
    shm.buf[0] = 0
    print(f"Avant : {shm.buf[0]}")

    p = multiprocessing.Process(target=worker_shm, args=(shm.name, shm.size))
    p.start()
    p.join()
    print(f"Après : {shm.buf[0]}")

    shm.close()
    shm.unlink()

---

## 7. Méthodes de démarrage : `fork`, `forkserver`, `spawn`

| Méthode | Copie mémoire | Sûreté | OS |
|---|---|---|---|
| `fork` | Copie du processus parent | Risque avec threads | Linux (défaut) |
| `forkserver` | Fork d'un serveur propre | Plus sûr | Linux |
| `spawn` | Nouveau processus from scratch | Le plus sûr | Windows/macOS (défaut) |

In [ ]:
import multiprocessing

print(f"Méthode actuelle : {multiprocessing.get_start_method()}")
print(f"Méthodes disponibles : {multiprocessing.get_all_start_methods()}")

In [ ]:
import multiprocessing

# On peut changer la méthode avec un context
ctx = multiprocessing.get_context("spawn")

def hello() -> None:
    print(f"Hello depuis PID {__import__('os').getpid()}")

if __name__ == "__main__":
    p = ctx.Process(target=hello)
    p.start()
    p.join()

**Recommandation Python 3.14 :** préférez `spawn` ou `forkserver`. `fork` est déprécié comme défaut dans Python 3.14 car il peut causer des deadlocks avec les threads.

---

## 8. Pièges courants

### 8.1. Oublier le guard `__main__`

Sans `if __name__ == "__main__"`, les processus `spawn` réimportent le module et relancent le code parent, créant une boucle infinie.

In [ ]:
# ❌ Mauvais (sur Windows/macOS) :
# p = multiprocessing.Process(target=fn)
# p.start()  # relance le script → boucle infinie

# ✅ Bon :
# if __name__ == "__main__":
#     p = multiprocessing.Process(target=fn)
#     p.start()
print("Toujours utiliser le guard if __name__ == '__main__'")

### 8.2. Objets non picklables

Tout ce qui est envoyé entre processus doit être picklable. Les lambdas, les connexions réseau et les objets avec état interne complexe ne sont pas picklables.

In [ ]:
import pickle

# ✅ Picklable
pickle.dumps((1, "hello", [1, 2, 3]))
print("tuple, str, list : picklables")

# ❌ Non picklable
try:
    pickle.dumps(lambda x: x + 1)
except pickle.PicklingError as e:
    print(f"Lambda : {e}")

### 8.3. Ne pas oublier `unlink()` avec `shared_memory`

Un bloc `SharedMemory` persiste au niveau de l'OS même après la fin du programme si `unlink()` n'est pas appelé. Utilisez `try/finally` ou un gestionnaire de contexte.

---

## 9. Synthèse

| Outil | Usage |
|---|---|
| `Process` | Un processus fils, contrôle fin |
| `Pool.map` | Paralléliser un map sur des données |
| `Pool.starmap` | map avec plusieurs arguments |
| `Pool.apply_async` | Soumission asynchrone |
| `Queue` | File FIFO inter-processus |
| `Pipe` | Communication bidirectionnelle |
| `Value` / `Array` | Données simples partagées (ctypes) |
| `shared_memory` | Bloc mémoire partagé haute performance |
| `spawn` | Méthode de démarrage recommandée |

**Règles d'or :**

1. Toujours protéger avec `if __name__ == "__main__"` ;
2. Tout objet échangé doit être picklable ;
3. Préférer `Pool` pour la simplicité, `Process` pour le contrôle ;
4. `shared_memory` pour les gros volumes de données.

---

## 10. Exercices

### Exercice 1 — Map parallèle de carrés *(facile)*

Utiliser `Pool.map` pour calculer les carrés des nombres de 1 à 1 000 000 avec 4 workers. Comparer le temps avec un `map` séquentiel.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Multiprocessing", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import multiprocessing
import time

def carre(x: int) -> int:
    return x * x

if __name__ == "__main__":
    N = 1_000_000
    data = list(range(1, N + 1))

    start = time.perf_counter()
    seq = list(map(carre, data))
    t_seq = time.perf_counter() - start

    start = time.perf_counter()
    with multiprocessing.Pool(4) as pool:
        par = pool.map(carre, data)
    t_par = time.perf_counter() - start

    print(f"Séquentiel : {t_seq:.3f}s")
    print(f"Pool(4)    : {t_par:.3f}s")
    assert seq == par
```

</details>

### Exercice 2 — Producteur / Consommateur multi-processus *(moyen)*

Implémenter un pattern producteur/consommateur avec `multiprocessing.Queue` :

- 2 producteurs qui envoient chacun 10 messages dans la queue.
- 1 consommateur qui lit et affiche les messages.
- Utiliser un sentinel `None` par producteur pour signaler la fin.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Multiprocessing", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import multiprocessing
import time

def producteur(q: multiprocessing.Queue, nom: str, n: int) -> None:
    for i in range(n):
        q.put(f"{nom}-{i}")
        time.sleep(0.01)
    q.put(None)

def consommateur(q: multiprocessing.Queue, nb_producteurs: int) -> None:
    fins = 0
    while fins < nb_producteurs:
        msg = q.get()
        if msg is None:
            fins += 1
        else:
            print(f"Consommé : {msg}")

if __name__ == "__main__":
    q = multiprocessing.Queue()
    p1 = multiprocessing.Process(target=producteur, args=(q, "P1", 10))
    p2 = multiprocessing.Process(target=producteur, args=(q, "P2", 10))
    c = multiprocessing.Process(target=consommateur, args=(q, 2))

    p1.start(); p2.start(); c.start()
    p1.join(); p2.join(); c.join()
    print("Terminé.")
```

</details>

### Exercice 3 — Compteur partagé avec `Value` *(moyen)*

Créer un compteur partagé (`Value('i', 0)`) protégé par un `Lock`. Lancer 8 processus qui incrémentent chacun 50 000 fois. Vérifier que le résultat est 400 000.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Multiprocessing", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import multiprocessing

def incrementer(compteur, lock, n: int) -> None:
    for _ in range(n):
        with lock:
            compteur.value += 1

if __name__ == "__main__":
    compteur = multiprocessing.Value('i', 0)
    lock = multiprocessing.Lock()
    procs = [
        multiprocessing.Process(target=incrementer, args=(compteur, lock, 50_000))
        for _ in range(8)
    ]
    for p in procs: p.start()
    for p in procs: p.join()
    print(f"Compteur : {compteur.value} (attendu : 400_000)")
    assert compteur.value == 400_000
```

</details>

### Exercice 4 — Map-Reduce parallèle *(difficile)*

Implémenter un compteur de mots distribué :

1. **Map** : chaque worker reçoit un texte et retourne un `dict[str, int]` de fréquences.
2. **Reduce** : le processus principal fusionne tous les dictionnaires.

Utiliser `Pool.map` pour la phase map. Tester avec 4 textes différents.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Multiprocessing", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import multiprocessing
from collections import Counter

def mapper(texte: str) -> dict[str, int]:
    mots = texte.lower().split()
    return dict(Counter(mots))

def reducer(resultats: list[dict[str, int]]) -> dict[str, int]:
    total: Counter[str] = Counter()
    for d in resultats:
        total.update(d)
    return dict(total)

if __name__ == "__main__":
    textes = [
        "Python est génial Python est rapide",
        "Le multiprocessing en Python est puissant",
        "Python permet le calcul parallèle",
        "Le GIL de Python est contourné par le multiprocessing",
    ]

    with multiprocessing.Pool(4) as pool:
        mapped = pool.map(mapper, textes)

    resultat = reducer(mapped)
    for mot, freq in sorted(resultat.items(), key=lambda x: -x[1])[:5]:
        print(f"  {mot}: {freq}")
```

</details>

---

## Ressources

- [docs Python — `multiprocessing`](https://docs.python.org/3/library/multiprocessing.html)
- [docs Python — `multiprocessing.shared_memory`](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
- [RealPython — multiprocessing](https://realpython.com/python-multiprocessing/)
- *High Performance Python* (M. Gorelick & I. Ozsvald), chapitre 9